# Session 10 lab: SQL for analysts

**Question of the day:** How do we ask a database the same questions in the language every employer uses?

In every step: **predict** first, run the query, **read** the result, then **change** one thing. For four queries you rebuild the answer in pandas, and a proof table records whether SQL and pandas agree. You keep this notebook as your query book.

Data: Lakeside Mart (fictional, generated), in a free SQLite database. Revenue describes a 5 percent sample of till orders.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
import os
import sqlite3
import pandas as pd

# If lakeside.db did not arrive with the data files, build it from the CSV files (same tables).
TABLES = ["branches", "products", "customers", "orders", "order_lines", "returns", "branch_weeks", "offer_test", "feedback"]
if not os.path.exists("lakeside.db"):
    _con = sqlite3.connect("lakeside.db")
    for t in TABLES:
        pd.read_csv(t + ".csv").to_sql(t, _con, index=False)
    _con.close()
con = sqlite3.connect("lakeside.db")

def q(sql):
    """Run a SQL query and return a DataFrame."""
    return pd.read_sql_query(sql, con)

orders = pd.read_csv("orders.csv")
branches = pd.read_csv("branches.csv")
customers = pd.read_csv("customers.csv")
returns = pd.read_csv("returns.csv")
products = pd.read_csv("products.csv")

# The proof table: one row for every SQL answer that you rebuild in pandas. Run this cell once.
proof = []
def compare(name, sql_value, pandas_value):
    proof.append(dict(query=name, sql=sql_value, pandas=pandas_value, match=bool(sql_value == pandas_value)))
def safe(f):
    try:
        return f()
    except Exception:
        return -1   # lets a check say NOT YET instead of stopping

---
## Step 1: Look at the database

A relational database is several tables linked by keys. SQLite keeps its list of tables in sqlite_master.

**PREDICT first.** How many tables does the database hold? How many records does orders have?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
tables = None    # SELECT name FROM sqlite_master WHERE type = 'table'
n_orders = None  # SELECT COUNT(*) AS n FROM orders, then take the number with .n[0]

# Automatic check
check('tables in the database', safe(lambda: len(tables)), 9)
check('records in orders', safe(lambda: n_orders), 30229)

**CHANGE one thing.** Run SELECT * FROM branches LIMIT 3. Which variable is the primary key?

> What I saw: 

---
## Step 2: SELECT, WHERE, ORDER BY, LIMIT

WHERE keeps records. ORDER BY sorts. LIMIT keeps the first few.

**PREDICT first.** How many orders were paid by Card at branch B06? What would change if you removed ORDER BY from the second query?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
n_card = None   # SELECT COUNT(*) AS n ... WHERE branch_id = 'B06' AND payment_method = 'Card'
top3 = None     # the three largest of those orders: order_id, order_value_ugx, ORDER BY ... DESC, LIMIT 3

# Automatic check
check('B06 card orders', safe(lambda: n_card), 257)
check('top three order values', safe(lambda: list(top3.order_value_ugx)), [1165400, 824600, 452800])

**READ the output.** Why do we add order_id after order_value_ugx in ORDER BY?

> My answer: 

---
## Step 3: GROUP BY, and the pandas twin

GROUP BY makes one result record per group. Then rebuild the same answer with groupby and compare.

**PREDICT first.** Which branch will have the largest revenue? Will pandas groupby give exactly the same number?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
by_branch = None   # SELECT branch_id, COUNT(*) AS orders, SUM(order_value_ugx) AS revenue ... GROUP BY ... ORDER BY revenue DESC
twin = None        # the same revenue per branch with groupby, sorted largest first
# then: compare('revenue of the top branch', sql_value, pandas_value)

# Automatic check
check('top branch revenue', safe(lambda: by_branch.revenue[0]), 283331305)
check('pandas twin agrees', safe(lambda: twin.iloc[0]), 283331305)

**READ the output.** Do the SQL and pandas answers agree? What does the match column of the proof table say?

> My answer: 

---
## Step 4: HAVING filters groups

WHERE filters records before grouping. HAVING filters groups after.

**PREDICT first.** Which return reasons have more than 450 returns?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
common = None   # GROUP BY reason, then HAVING COUNT(*) > 450

# Automatic check
check('reasons above 450', safe(lambda: len(common)), 3)
check('largest reason count', safe(lambda: common.n[0]), 683)

**CHANGE one thing.** Change 450 to 470. Which reasons remain? Then write the same filter with WHERE and see the error message.

> What I saw: 

---
## Step 5: INNER JOIN with aliases

ON names the join key. An alias such as o or b is a short name for a table.

**PREDICT first.** Joining orders to branches adds the region. Will the row count of the join change? Which region leads?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
by_region = None    # JOIN orders AS o to branches AS b ON the branch_id, GROUP BY b.region, SUM, ORDER BY revenue DESC
twin_region = None  # the pandas twin: merge (validate='m:1'), groupby region, sum, sort
# then compare(...)

# Automatic check
check('top region revenue', safe(lambda: by_region.revenue[0]), 679043495)
check('top region orders', safe(lambda: by_region.orders[0]), 14060)

**READ the output.** Add up the four region revenues. What number should you get? Why?

> My answer: 

---
## Step 6: LEFT JOIN keeps the walk-ins

A left join keeps every record of the left table. Records with no match get NULL.

**PREDICT first.** Count orders by loyalty tier with a LEFT JOIN to customers. How many orders will have a NULL tier?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
tiers = None       # LEFT JOIN customers AS c ON the customer_id, GROUP BY c.loyalty_tier, COUNT(*)
n_inner = None     # the same join with INNER JOIN, just COUNT(*)
null_tier = None   # the number of orders whose tier is NULL

# Automatic check
check('orders with a NULL tier', safe(lambda: null_tier), 9171)
check('rows of the inner join', safe(lambda: n_inner), 21058)

**CHANGE one thing.** Replace LEFT JOIN with INNER JOIN in the first query. Which group disappears?

> What I saw: 

---
## Step 7: NULL and three-valued logic

A test on NULL is UNKNOWN, and WHERE keeps only TRUE. Use IS NULL to find missing values.

**PREDICT first.** How many orders have customer_id <> 'C0495'? Will the walk-in orders be among them?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
not_c0495 = None   # COUNT(*) ... WHERE customer_id <> 'C0495'
walk_in = None     # COUNT(*) ... WHERE customer_id IS NULL

# Automatic check
check('not C0495', safe(lambda: not_c0495), 21050)
check('walk-ins', safe(lambda: walk_in), 9171)

**READ the output.** Add the two counts and the number of orders that equal C0495. Do they make every order?

> My answer: 

---
## Step 8: CASE WHEN, and the pandas twin

CASE returns the value of the first condition that is true.

**PREDICT first.** Bands: Small under 20,000, Medium under 100,000, Large the rest. Which band will have the most orders?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
bands = None   # CASE WHEN ... THEN ... ELSE ... END AS band, COUNT(*), GROUP BY band; keep band and orders (set_index('band').orders)
cut = None     # the pandas twin: pd.cut on order_value_ugx with three labels, then value_counts
# then compare(...)

# Automatic check
check('Medium orders', safe(lambda: bands['Medium']), 14486)
check('Large orders', safe(lambda: bands['Large']), 2214)

**CHANGE one thing.** Move the Small limit from 20000 to 10000. Which band changes most? Then add WHERE band = 'Small' (an alias in WHERE) to the query. SQLite accepts it, but the SQL standard and many databases refuse it. How would you write it so that it runs everywhere?

> What I saw: 

---
## Step 9: Subquery and CTE

A subquery is a query in brackets that supplies a value. A CTE (WITH) names a step so the query reads in order.

**PREDICT first.** How many of the 150 products cost more than the average unit price?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
n_dear = None   # COUNT(*) FROM products WHERE unit_price_ugx > (a subquery with AVG(unit_price_ugx))
twin_n = None   # the pandas twin with mean()
cte = None      # WITH by_branch AS (...) SELECT COUNT(*) AS branches, MAX(revenue) AS best FROM by_branch
# then compare(...)

# Automatic check
check('products above average', safe(lambda: n_dear), 20)
check('best branch revenue from the CTE', safe(lambda: cte.best[0]), 283331305)

**READ the output.** Which is easier to read, the subquery or the CTE? Write one sentence.

> My answer: 

---
## Step 10: Keep: the proof table

Every row shows the SQL answer, the pandas answer and whether they match. A False means a mistake in one of them.

In [ ]:
proof_table = None   # pd.DataFrame(proof), then print it
all_match = None     # True when every row of the match column is True

# Automatic check
check('proof table has four rows', safe(lambda: len(proof_table)), 4)
check('every SQL answer matches pandas', safe(lambda: int(all_match)), 1)

**Keep this notebook** as your query book. Save a copy to your Drive (File, Save a copy in Drive). Bring one question from your own database to Session 11.

In [ ]:
show_checks()